# Danh mục món từ thực đơn khu Thủ Đức cũ

Pipeline khảo sát độc lập, theo yêu cầu chủ dự án. Chưa là seed, coverage hay offering đã review của GM-03/GM-27. Ưu tiên tên món thực tế, giữ nguyên bằng chứng menu. Run All từ trên xuống; dữ liệu và báo cáo tự tạo dưới `datasets/thu-duc-menus/`. Không dùng dataset công thức.


## 1. Cấu hình và môi trường

Mặc định dùng snapshot đã có; chỉ tải nguồn còn thiếu. `REFRESH=True` tải bản mới. `ALLOW_NETWORK=False` chạy hoàn toàn từ snapshot. Python cần pandas/lxml có sẵn; không cài dependency ứng dụng. Nguồn và phạm vi xem ở cell sau. Giữ HTTP snapshot; không tải ảnh hoặc đăng nhập.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
from urllib.parse import urlparse, urljoin
from urllib.error import URLError, HTTPError
import hashlib, json, platform, re, time, unicodedata
from collections import Counter, defaultdict
import pandas as pd
import lxml
from lxml import html

REFRESH = False
ALLOW_NETWORK = True
DATA_PREPARATION_DIR = None  # Path('/.../data-preparation') nếu kernel nằm ngoài repo
REQUEST_DELAY_SECONDS = 1.0
TIMEOUT_SECONDS = 25
runner_options = globals().get('NOTEBOOK_OPTIONS', {})
ALLOW_NETWORK = ALLOW_NETWORK and not runner_options.get('offline', False)
REFRESH = REFRESH or runner_options.get('refresh', False)
if DATA_PREPARATION_DIR is None:
    candidates = [p for parent in (Path.cwd(), *Path.cwd().parents)
                  for p in (parent, parent / 'data-preparation')]
    DATA_PREPARATION_DIR = next((p for p in candidates if
        (p / 'notebooks/prepare_thu_duc_menus.ipynb').is_file()), None)
if DATA_PREPARATION_DIR is None:
    raise FileNotFoundError('Đặt DATA_PREPARATION_DIR vào thư mục data-preparation của project.')
DATA_PREPARATION_DIR = Path(DATA_PREPARATION_DIR).resolve()
DATASET_DIR = DATA_PREPARATION_DIR / 'datasets/thu-duc-menus'
RAW_DIR, PROCESSED_DIR, REPORTS_DIR = [DATASET_DIR / p for p in ('raw', 'processed', 'reports')]
for folder in (RAW_DIR / 'html', PROCESSED_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RAW_DIR / 'source_manifest.json'
OUTPUT_PATHS = {name: PROCESSED_DIR / (name + '.csv') for name in
                ('menu_items_raw', 'dish_catalogue', 'menu_dish_mapping')}
def digest(data): return hashlib.sha256(data).hexdigest()
def stable_id(prefix, *parts):
    return prefix + '_' + digest(json.dumps(parts, ensure_ascii=False).encode())[:20]
def text(node): return ' '.join(node.text_content().split()) if node is not None else ''
def norm(value):
    value = unicodedata.normalize('NFC', str(value)).casefold()
    return ' '.join(re.sub(r'[^\w\s]', ' ', value, flags=re.UNICODE).split())
def as_json(value): return json.dumps(value, ensure_ascii=False, sort_keys=True)
pd.set_option('display.max_colwidth', 100)
display(pd.DataFrame([{'Python': platform.python_version(), 'pandas': pd.__version__,
                      'lxml': lxml.__version__, 'dataset': str(DATASET_DIR),
                      'refresh': REFRESH, 'network': ALLOW_NETWORK}]))


## 2. Nguồn khảo sát và phạm vi

Danh sách URL khởi đầu nằm trong `config/thu_duc_sources.json`. Có nhóm quán, cụm khảo sát và token địa chỉ để phát hiện tải nhầm chi nhánh. Có thể thêm URL đã tìm được vào config rồi Run All; không tự tuyên bố đã tìm hết quán. Điểm/radius người dùng và polygon coverage chưa được thiết lập ở bước này. Không dùng khoảng cách giao hàng trên nền tảng làm tọa độ khảo sát.


In [ ]:
source_config = json.loads((DATA_PREPARATION_DIR / 'config/thu_duc_sources.json').read_text('utf-8'))
sources = source_config['sources']
assert sources and len({s['url'] for s in sources}) == len(sources), 'URL trùng/rỗng'
allowed_hosts = {'befood': 'food.be.com.vn', 'shopeefood': 'shopeefood.vn'}
for source in sources:
    parsed = urlparse(source['url'])
    assert parsed.scheme == 'https' and parsed.hostname == allowed_hosts[source['platform']]
    source['source_id'] = stable_id('src', source['url'])
    # Một nguồn tương ứng một chi nhánh; khi thêm nền tảng khác cho cùng quán,
    # đặt cùng branch_key sau khi kiểm địa chỉ để không đếm đôi chi nhánh.
    source['venue_id'] = stable_id('venue', source.get('branch_key', source['url']))
print(source_config['scope'])
display(pd.DataFrame(sources))
display(pd.DataFrame(sources).groupby(['survey_cluster', 'discovery_group']).size().rename('sources').reset_index())


## 3. Tải HTML và lưu snapshot có checksum

Tải lần lượt với timeout và khoảng nghỉ, giữ URL yêu cầu/URL trả về, UTC thực tế, kích thước, SHA-256. Lỗi một nguồn không làm mất các nguồn khác. Snapshot bị sửa phải báo lỗi; refresh lỗi vẫn dùng snapshot cũ nếu có, ghi rõ fallback và giữ ngày cũ. HTTP 403/429 không vượt chặn. ShopeeFood có thể chỉ trả vỏ trang: báo needs_render, chưa coi là menu rỗng. Không gọi API nội bộ.


In [ ]:
manifest = json.loads(MANIFEST_PATH.read_text('utf-8')) if MANIFEST_PATH.exists() else {'schema_version': 1, 'sources': {}}
snapshots, fetch_log = {}, []
for source in sources:
    sid = source['source_id']
    cached = manifest['sources'].get(sid)
    path = RAW_DIR / 'html' / (sid + '.html')
    if cached and (not path.exists() or digest(path.read_bytes()) != cached['sha256']):
        raise ValueError(f'Snapshot không đúng manifest: {path}; khôi phục hoặc refresh có chủ đích.')
    state, error = 'cached', ''
    if REFRESH or not cached:
        if not ALLOW_NETWORK:
            state, error = 'missing_snapshot', 'Network disabled'
        else:
            try:
                request = Request(source['url'], headers={'User-Agent': 'AnyfoodDataResearch/0.1 (public menu survey)'})
                with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
                    blob = response.read(8_000_001)
                    if len(blob) > 8_000_000:
                        raise ValueError('HTML vượt giới hạn 8 MB')
                    resolved_url = response.geturl()
                    if urlparse(resolved_url).hostname != allowed_hosts[source['platform']]:
                        raise ValueError('Redirect sang host khác')
                    content_type = response.headers.get('Content-Type', '')
                    if 'html' not in content_type.lower():
                        raise ValueError('Nguồn không trả HTML')
                cached = dict(source_url=source['url'], resolved_url=resolved_url,
                              fetched_at=datetime.now(timezone.utc).isoformat(), sha256=digest(blob),
                              bytes=len(blob), content_type=content_type, snapshot_file=path.name)
                temp = path.with_suffix('.tmp')
                temp.write_bytes(blob)
                temp.replace(path)
                manifest['sources'][sid] = cached
                MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
                state = 'fetched'
            except (URLError, HTTPError, TimeoutError, OSError, ValueError) as exc:
                state, error = 'fetch_error', f'{type(exc).__name__}: {exc}'
            finally:
                time.sleep(REQUEST_DELAY_SECONDS)
    if cached:
        snapshots[sid] = (path.read_bytes(), cached)
        if error: state = 'cached_after_fetch_error'
    fetch_log.append(dict(source_id=sid, source_url=source['url'], fetch_state=state, error=error,
                          fetched_at=cached['fetched_at'] if cached else ''))
    print(source['platform'], source['url'].rsplit('/',1)[-1], state)
display(pd.DataFrame(fetch_log))


## 4. Trích xuất toàn bộ mục menu có trong snapshot

Adapter beFood đọc section menu HTML, gộp mục “Món phải thử” trùng với nhóm chính. Tên/mô tả/giá gốc được giữ; ghép ảnh từ card theo tên và giá khi không nhập nhằng. Nhiều ID cùng tên vẫn giữ từng ID. Giữ nhóm nguồn, HTML từng mục, URL ảnh/srcset, ID nền tảng, địa chỉ và geo nếu nguồn có. Chỉ thu metadata Restaurant/menu; không lấy nội dung người đánh giá.

Số mục menu trong HTML và số card có ảnh được báo riêng: không hứa web chứa toàn menu ứng dụng. Disabled có thể vì quán đóng cửa, nên không suy thành hết món. URL ảnh chưa xác nhận quyền sử dụng. Adapter ShopeeFood báo cần render khi HTML chưa chứa menu; chưa hỗ trợ lấy menu tự động bằng trình duyệt trong notebook này.


In [ ]:
def jsonld_restaurant(tree):
    for script in tree.xpath('//script[@type="application/ld+json"]'):
        try: obj = json.loads(script.text or '{}')
        except json.JSONDecodeError: continue
        nodes = obj if isinstance(obj, list) else [obj]
        for node in nodes:
            for candidate in node.get('@graph', [node]):
                if candidate.get('@type') == 'Restaurant': return candidate
    return {}

def parse_menu(blob, source):
    tree = html.fromstring(blob.decode('utf-8'))
    ld = jsonld_restaurant(tree)
    headings = tree.xpath('//h1')
    name = text(headings[0]) if headings else ld.get('name', '')
    address = ld.get('address', {}).get('streetAddress', '')
    if not address and headings:
        paragraphs = headings[0].getparent().getparent().xpath('.//p')
        address = text(paragraphs[0]) if paragraphs else ''
    geo = ld.get('geo', {})
    venue = dict(venue_name=name, address=address, latitude=geo.get('latitude', ''),
                 longitude=geo.get('longitude', ''), restaurant_image_url=ld.get('image', ''),
                 opening_hours_source_json=as_json(ld.get('openingHoursSpecification', [])),
                 restaurant_source_json=as_json({k: ld[k] for k in
                    ('name','address','geo','image','openingHoursSpecification','servesCuisine') if k in ld}))
    if source['platform'] != 'befood':
        return [], venue, {'parse_state':'needs_render', 'seo_occurrences':0, 'visible_cards':0,
                           'note':'HTML trực tiếp chưa có adapter menu; không coi là không có món.'}
    sections = tree.xpath('//section[@aria-label="Thực đơn của nhà hàng"]')
    if not sections:
        return [], venue, {'parse_state':'menu_missing', 'seo_occurrences':0, 'visible_cards':0,
                           'note':'Không thấy section menu; kiểm layout/nguồn trước khi dùng.'}
    cards = defaultdict(dict)
    card_records = {}
    for card in tree.xpath('//article[@data-dish-id]'):
        hs = card.xpath('.//h3')
        prices = card.xpath('.//span[contains(text(), "₫")]')
        if not hs or not prices: continue
        price = re.sub(r'[^0-9]', '', text(prices[-1]))
        key = (norm(text(hs[0])), price)
        card_id = card.get('data-dish-id')
        imgs = card.xpath('.//img')
        images = [dict(url=urljoin(source['url'], i.get('src', '')),
                       srcset=i.get('srcset', ''), alt=i.get('alt', '')) for i in imgs if i.get('src')]
        card_records[card_id] = dict(platform_item_id=card_id, images=images, ui_name=text(hs[0]),
            ui_price_labels=[text(p) for p in prices],
            aria_disabled=card.get('aria-disabled', ''), card_html=html.tostring(card, encoding='unicode'))
        cards[key][card_id] = card_records[card_id]
    collected = {}
    seo_count = 0
    for section_index, section in enumerate(sections):
        for group_index, group in enumerate(section.xpath('./div[h3]')):
            group_name = text(group.xpath('./h3')[0])
            for article_index, article in enumerate(group.xpath('./article[h4]')):
                seo_count += 1
                raw_name = text(article.xpath('./h4')[0])
                prices = article.xpath('./data')
                price = prices[0].get('value', '') if prices else ''
                desc = ' '.join(text(p) for p in article.xpath('./p'))
                candidates = list(cards.get((norm(raw_name), price), {}).values())
                metadata_match = 'unique_name_price'
                if not candidates:
                    candidates = [c for c in card_records.values() if norm(c['ui_name']) == norm(raw_name)]
                    metadata_match = 'unique_name_price_difference'
                if not candidates:
                    candidates = [dict(platform_item_id='', images=[], aria_disabled='', card_html='', ui_price_labels=[])]
                    metadata_match = 'seo_only'
                for card in candidates:
                    item_key = card['platform_item_id'] or stable_id('seo', raw_name, price, desc)
                    occurrence = dict(occurrence_id=stable_id('occ', source['source_id'], section_index, group_index, article_index),
                                      origin='seo_menu', group=group_name, name=raw_name, description=desc,
                                      price_raw=text(prices[0]) if prices else '', price_value=price,
                                      article_html=html.tostring(article, encoding='unicode'))
                    if item_key not in collected:
                        collected[item_key] = dict(menu_item_id=stable_id('menu', source['source_id'], item_key),
                            platform_item_id=card['platform_item_id'], menu_name_raw=raw_name,
                            metadata_match_status='ambiguous_name_price' if len(candidates)>1 else metadata_match,
                            description_raw=desc, price_raw=occurrence['price_raw'], price_value=price,
                            currency='VND' if price else '', price_unit='menu_item_unspecified',
                            image_url=card['images'][0]['url'] if card['images'] else '',
                            ui_price_labels_json=as_json(card['ui_price_labels']),
                            images_json=as_json(card['images']), image_usage_rights='unknown',
                            availability='unknown', ui_disabled=card['aria_disabled'],
                            card_html=card['card_html'], occurrences=[])
                    collected[item_key]['occurrences'].append(occurrence)
    matched_card_ids = {v['platform_item_id'] for v in collected.values() if v['platform_item_id']}
    for card_id, card in card_records.items():
        if card_id in matched_card_ids: continue
        # Keep cards missing from the SEO menu instead of silently dropping them.
        occurrence = dict(occurrence_id=stable_id('occ_ui', source['source_id'], card_id),
            origin='ui_card', group='UI card chưa ghép được nhóm', name=card['ui_name'],
            description='', price_raw='', price_value='', article_html=card['card_html'])
        collected[card_id] = dict(menu_item_id=stable_id('menu', source['source_id'], card_id),
            platform_item_id=card_id, menu_name_raw=card['ui_name'], description_raw='',
            metadata_match_status='ui_only_needs_review', price_raw='', price_value='',
            ui_price_labels_json=as_json(card['ui_price_labels']), currency='VND',
            price_unit='menu_item_unspecified', image_url=card['images'][0]['url'] if card['images'] else '',
            images_json=as_json(card['images']), image_usage_rights='unknown',
            availability='unknown', ui_disabled=card['aria_disabled'],
            card_html=card['card_html'], occurrences=[occurrence])
    rows = []
    for item in collected.values():
        occurrences = item.pop('occurrences')
        item['menu_groups_json'] = as_json(sorted({o['group'] for o in occurrences}))
        item['source_occurrences_json'] = as_json(occurrences)
        item['source_occurrence_count'] = len(occurrences)
        rows.append(item)
    return rows, venue, dict(parse_state='menu_extracted' if rows else 'menu_empty',
        seo_occurrences=seo_count, visible_cards=sum(len(v) for v in cards.values()),
        unmatched_cards_kept=len(card_records)-len(matched_card_ids),
        note='Đọc menu có trong HTML; chưa xác nhận toàn menu app, lịch bán hay tồn món.')

menu_rows, source_report = [], []
for source in sources:
    sid = source['source_id']
    fetch = next(f for f in fetch_log if f['source_id'] == sid)
    if sid not in snapshots:
        source_report.append({**source, **fetch, 'parse_state':'not_fetched', 'menu_items':0})
        continue
    blob, meta = snapshots[sid]
    rows, venue, diagnostics = parse_menu(blob, source)
    scope_match = bool(venue['address']) and norm(source['expected_address_token']) in norm(venue['address'])
    scope_status = 'selected_address_matches' if scope_match else 'needs_address_review'
    for row in rows:
        menu_rows.append({**row, **venue, 'source_id':sid, 'venue_id':source['venue_id'],
                          'platform':source['platform'], 'source_url':source['url'],
                          'survey_cluster':source['survey_cluster'], 'scope_status':scope_status,
                          'checked_at':meta['fetched_at'], 'snapshot_sha256':meta['sha256'],
                          'review_status':'draft', 'service_mode':'delivery_platform_menu'})
    source_report.append({**source, **fetch, **venue, **diagnostics, 'scope_status':scope_status,
                          'menu_items':len(rows), 'items_with_images':sum(bool(r['image_url']) for r in rows)})
if not menu_rows:
    raise RuntimeError('Chưa lấy được menu nào; xem fetch_log, mạng và adapter. Không xuất danh mục rỗng như thành công.')
raw_df = pd.DataFrame(menu_rows).sort_values('menu_item_id').reset_index(drop=True)
source_df = pd.DataFrame(source_report)
assert raw_df.menu_item_id.is_unique
display(source_df[['platform','survey_cluster','venue_name','parse_state','scope_status','menu_items','items_with_images']])
display(raw_df[['menu_name_raw','description_raw','price_raw','image_url','menu_groups_json','venue_name']].head(25))
display(raw_df.groupby('platform').agg(items=('menu_item_id','size'), venues=('venue_id','nunique')))
display(raw_df.isna().sum().to_frame('missing'))


## 5. Chuẩn hóa tên và bảng quy tắc phân loại

Giữ dấu Unicode; không fuzzy-match hay suy tên món từ nguyên liệu. Chỉ sửa chính tả trong SPELLING; tách hậu tố kích cỡ rõ và số lượng một mục mì/burger/bánh cuộn. Tên gốc và giá đơn vị luôn được giữ. Tên bún có topping, bánh bao nhân và bánh cuộn là ứng viên; không nhận “Bún”, “Cơm”, “Phở” đơn lẻ.

Nhóm đồ uống/món thêm/khai vị/món ăn chơi loại khỏi danh mục bữa chính. Combo và lẩu/khẩu phần nhóm giữ needs_review. Nhóm nguồn ghi rõ Cơm Chay/Bún Chay cho phép thêm “chay” vào tên chuẩn và ghi lý do; chỉ tên quán chay không đủ để tự chốt. Nhãn này phản ánh menu nguồn, chưa xác minh thành phần hay an toàn dị ứng.

Bảng bên dưới và mapping ghi lại mọi quyết định. Dữ liệu vẫn draft, chưa chứng minh bán buổi trưa hoặc đang còn món.


In [ ]:
SPELLING = {'mỳ':'mì', 'xốt':'sốt', 'hambuger':'hamburger'}
ROOTS = [('hủ tiếu bò kho','hủ tiếu bò kho','hủ tiếu'), ('bún thịt nướng','bún thịt nướng','bún'),
         ('bún đậu','bún đậu','bún'), ('bún riêu','bún riêu','bún'), ('bún bò','bún bò','bún'),
         ('bánh mì','bánh mì','bánh mì'), ('bánh cuốn','bánh cuốn','bánh cuốn'),
         ('bánh ướt','bánh ướt','bánh ướt'), ('bánh canh','bánh canh','bánh canh'),
         ('cơm tấm','cơm tấm','cơm'), ('cơm','cơm','cơm'), ('phở','phở','phở'),
         ('hủ tiếu','hủ tiếu','hủ tiếu'), ('mì','mì','mì'), ('miến','miến','miến'),
         ('nui','nui','nui'), ('xôi','xôi','xôi'), ('cháo','cháo','cháo'),
         ('salad','salad','salad'), ('burger','burger','burger'), ('hamburger','burger','burger'),
         ('nem nướng','nem nướng','nem nướng'), ('bò né','bò né','bò né'),
         ('bánh bao','bánh bao','bánh bao'), ('bánh cuộn','bánh cuộn','bánh cuộn'),
         ('bún','bún','bún')]
DRINK = r'^(?:trà|cà phê|cafe|coffee|matcha|cacao|ca cao|sữa|nước|soda|sôđa|sô đa|sinh tố|ép|pepsi|pesi|coca|7up|7 úp|sting|bạc xỉu|kombucha|cold brew|đá me)\b'
SIDE = r'^(?:\d+\s+)?(?:cơm (?:thêm|trắng)|mì thêm|phở thêm|bún thêm|miến thêm|rau|sốt|mắm|quẩy|trứng|chén|canh|bánh mì thêm)\b'
SIDE_EXACT = {'bún','mì gói','bánh mì','đậu hũ','thịt bằm','xúc xích','phô mai lát','cá viên','chả cốm','dồi sụn'}
NON_MEAL = r'^(?:chè|kem|bánh (?:kem|su|choux|chuối|trứng|quy)|thạch|trân châu|xoài|ngô|khoai tây)\b'
rule_table = pd.DataFrame([
    ('unicode_punctuation','Tự gộp tên tương đương về ký tự; giữ dấu.'),
    ('spelling_alias','Chỉ thay từ trong SPELLING; giữ tên gốc.'),
    ('size_suffix','Tách nhỏ/lớn/size M/L ở cuối; không xóa topping/cách nấu.'),
    ('drink_side_dessert','Giữ dữ liệu nhưng không đưa vào danh mục bữa chính.'),
    ('combo_or_group','Combo/set/lẩu/khẩu phần nhóm cần rà; không quy giá/người.'),
    ('chay_context','Không gộp món quán chay không ghi rõ vào món thịt.'),
    ('main_prefix','Tên bắt đầu bằng món chính rõ → ứng viên, vẫn draft.'),
    ('explicit_menu_group','Đồ uống/món thêm/khai vị/món ăn chơi theo nhóm nguồn bị loại.'),
    ('explicit_chay_group','Bổ sung chay khi nhóm cơm/bún ghi rõ chay; vẫn chưa xác minh thành phần.'),
    ('unit_one_prefix','Bỏ 1 trước mì/burger/bánh cuộn; giữ giá menu và đơn vị chưa rõ.'),
    ('unknown','Tên rút gọn hoặc nghĩa nhập nhằng → needs_review.')], columns=['rule','meaning'])
display(rule_table)

def classify(row):
    raw = row['menu_name_raw']
    cleaned = unicodedata.normalize('NFC', raw).casefold()
    alias_changes = []
    for old, new in SPELLING.items():
        updated = re.sub(r'\b'+old+r'\b', new, cleaned)
        if updated != cleaned: alias_changes.append(old+' → '+new)
        cleaned = updated
    size_match = re.search(r'\s*(?:[-–]?\s*\b(?:nhỏ|vừa|lớn)|\bsize\s+(?:xl|[sml]))\s*$', cleaned)
    size = size_match.group().strip(' -–') if size_match else ''
    if size_match: cleaned = cleaned[:size_match.start()]
    # Chỉ bỏ số lượng 1 trước tên bữa chính rõ; không quy đổi giá theo người.
    quantity_match = re.match(r'^0?1\s+(?=(?:mì|burger|bánh cuộn)\b)', norm(cleaned))
    if quantity_match:
        cleaned = re.sub(r'^\s*0?1\s+', '', cleaned)
        alias_changes.append('Bỏ số lượng một mục menu khỏi tên món; giá vẫn theo mục gốc.')
    normalized = norm(cleaned)
    groups = ' '.join(json.loads(row['menu_groups_json']))
    normalized_groups = [norm(g) for g in json.loads(row['menu_groups_json'])]
    drink_group = any(g in {'nước uống', 'đồ uống', 'thức uống', 'nước', 'nước giải khát'}
                      for g in normalized_groups)
    side_group = any(g in {'thêm', 'topping', 'món thêm', 'khai vị', 'team ăn kèm', 'món ăn chơi'}
                     or g.startswith('món thêm ') for g in normalized_groups)
    explicit_chay_group = any(g.startswith(('cơm chay ', 'bún chay ')) or g in {'cơm chay', 'bún chay'}
                              for g in normalized_groups)
    context_chay = 'chay' in norm(row['venue_name']+' '+groups)
    root_name, category = '', ''
    for prefix, root, cat in ROOTS:
        if normalized == prefix or normalized.startswith(prefix+' '):
            root_name, category = root, cat
            break
    status, kind, reason = 'needs_review', 'unknown', 'Tên chưa đủ rõ để coi là một bữa chính.'
    if re.search(r'\b(combo|set|com bo|xô|mẹt)\b', normalized) or '+' in raw:
        kind, reason = 'combo', 'Giữ combo gốc; cần rà món thành phần và khẩu phần.'
    elif re.match(DRINK, normalized) or drink_group:
        status, kind, reason = 'excluded', 'drink', 'Đồ uống.'
    elif (re.match(SIDE, normalized) or normalized in SIDE_EXACT or normalized.endswith(' thêm') or
          ('phần thêm' in norm(groups)) or ('món ăn thêm' in norm(groups)) or side_group):
        status, kind, reason = 'excluded', 'side', 'Món thêm/nguyên liệu; không là bữa chính độc lập.'
    elif re.match(NON_MEAL, normalized):
        status, kind, reason = 'excluded', 'dessert_or_side', 'Tráng miệng hoặc món phụ theo tên.'
    elif normalized.startswith('lẩu') or re.search(r'\b(nguyên con|nửa con|nữa ký|nửa ký|\d+ người)\b', normalized):
        kind, reason = 'group_portion', 'Cần kiểm khẩu phần nhóm và giá đơn vị.'
    elif context_chay and 'chay' not in normalized:
        if explicit_chay_group and root_name and (normalized != root_name or normalized in {'bún bò','bún riêu'}):
            normalized += ' chay'
            status, kind = 'candidate', 'main_chay_context'
            reason = 'Nhóm menu nguồn ghi rõ cơm/bún chay; bổ sung chay vào tên để không ghép món thịt. Chưa xác minh thành phần.'
            alias_changes.append('Bổ sung chay theo tên nhóm menu nguồn.')
        else:
            kind, reason = 'chay_context', 'Menu quán/nhóm chay: cần xác nhận tên chuẩn riêng, không ghép món thịt.'
    elif root_name and normalized != root_name:
        status, kind, reason = 'candidate', 'main', 'Tên món chính rõ; ứng viên danh mục, chưa review.'
    elif normalized in {'bún bò','bún riêu','bún thịt nướng','bánh cuốn','bánh ướt','bún đậu','bò né','nem nướng'}:
        status, kind, reason = 'candidate', 'main', 'Tên món hoàn chỉnh; ứng viên danh mục.'
    if row['scope_status'] != 'selected_address_matches':
        status, reason = 'needs_review', 'Địa chỉ nguồn chưa khớp khu khảo sát.'
    if row.get('metadata_match_status') in {'ambiguous_name_price', 'ui_only_needs_review'}:
        status, reason = 'needs_review', 'Nhiều ID menu cùng tên/giá; giữ riêng nhưng chưa chốt metadata.'
    return dict(normalized_name=normalized, canonical_name=normalized.capitalize(), root_dish=root_name,
                category=category, variant=normalized[len(root_name):].strip() if root_name and normalized.startswith(root_name) else normalized,
                size_variant=size, alias_changes_json=as_json(alias_changes), context_chay=context_chay,
                mapping_status=status, item_kind=kind, mapping_reason=reason,
                dish_id=stable_id('dish', normalized) if status == 'candidate' else '')
mapping_df = pd.DataFrame([{**{k:r[k] for k in ('menu_item_id','venue_id','menu_name_raw','source_url','checked_at')},
                            **classify(r)} for r in menu_rows]).sort_values('menu_item_id').reset_index(drop=True)
display(mapping_df[['menu_name_raw','canonical_name','root_dish','size_variant','mapping_status','mapping_reason']].head(40))
display(mapping_df.groupby(['mapping_status','item_kind']).size().rename('items').reset_index())
with pd.option_context('display.max_rows', None):
    display(mapping_df[mapping_df.mapping_status == 'needs_review'][['menu_name_raw','mapping_reason','source_url']])


## 6. Tổng hợp danh mục ứng viên và độ bao phủ mẫu

Mỗi tên chuẩn một ID ổn định, giữ mọi alias/tên gốc, biến thể kích cỡ và nguồn/quán. Ưu tiên theo số chi nhánh trong mẫu; không dùng số này làm lượng người thích. Danh mục không trộn giá thấp nhất với ảnh/mô tả quán khác thành một offering. Metadata từng menu nằm trong mapping/raw; ảnh/mô tả catalogue giữ tham chiếu đúng mục nguồn. Thiếu cuisine/origin/temperature/flavor để unknown. Mục chưa kết luận có trong bảng mapping để không bị mất.


In [ ]:
joined = raw_df.merge(mapping_df[['menu_item_id','dish_id','mapping_status','canonical_name','root_dish','category','size_variant']],
                      on='menu_item_id', validate='one_to_one')
catalogue_rows=[]
for dish_id, matches in joined[joined.mapping_status == 'candidate'].groupby('dish_id', sort=True):
    first = matches.sort_values('menu_item_id').iloc[0]
    catalogue_rows.append(dict(dish_id=dish_id, canonical_name=first.canonical_name,
        root_dish=first.root_dish, category=first.category,
        aliases_json=as_json(sorted(set(matches.menu_name_raw))),
        size_variants_json=as_json(sorted(set(matches.size_variant)-{''})),
        venue_count=matches.venue_id.nunique(), menu_item_count=len(matches),
        venue_ids_json=as_json(sorted(set(matches.venue_id))),
        source_urls_json=as_json(sorted(set(matches.source_url))),
        descriptions_json=as_json([dict(menu_item_id=r.menu_item_id, text=r.description_raw, source_url=r.source_url)
                                  for r in matches.itertuples() if r.description_raw]),
        images_json=as_json([dict(menu_item_id=r.menu_item_id, url=r.image_url, source_url=r.source_url,
                                 usage_rights='unknown') for r in matches.itertuples() if r.image_url]),
        menu_item_ids_json=as_json(sorted(matches.menu_item_id)),
        cuisine='unknown', origin='unknown', temperature='unknown', flavor='unknown',
        meal_slots='needs_review', review_status='draft', dataset_version='thu-duc-menu-survey-v1'))
CATALOGUE_COLUMNS = ['dish_id','canonical_name','root_dish','category','aliases_json','size_variants_json',
    'venue_count','menu_item_count','venue_ids_json','source_urls_json','descriptions_json','images_json',
    'menu_item_ids_json','cuisine','origin','temperature','flavor','meal_slots','review_status','dataset_version']
catalogue_df = pd.DataFrame(catalogue_rows, columns=CATALOGUE_COLUMNS).sort_values(
    ['venue_count','canonical_name'], ascending=[False,True]).reset_index(drop=True)
stats = dict(sources_selected=len(sources), sources_with_menu=int((source_df.menu_items > 0).sum()),
             venues_with_menu=raw_df.venue_id.nunique(), raw_menu_items=len(raw_df),
             catalogue_candidates=len(catalogue_df), mapping_counts=mapping_df.mapping_status.value_counts().to_dict(),
             items_with_images=int(raw_df.image_url.ne('').sum()),
             sources_needing_render=int(source_df.parse_state.eq('needs_render').sum()),
             selected_sources_without_menu=int(source_df.menu_items.eq(0).sum()),
             scope='Selected public-address branches in old Thu Duc; no full-area completeness claim')
display(pd.DataFrame([stats]))
with pd.option_context('display.max_rows', None):
    display(catalogue_df[['canonical_name','root_dish','category','venue_count','menu_item_count']])
display(source_df.groupby(['survey_cluster','parse_state']).size().rename('sources').reset_index())
source_df.to_json(REPORTS_DIR / 'source_coverage.json', orient='records', force_ascii=False, indent=2)
(REPORTS_DIR / 'quality_report.json').write_text(json.dumps(stats,ensure_ascii=False,indent=2)+'\n', encoding='utf-8')


## 7. Kiểm chứng và xuất CSV

Kiểm liên kết 1–1 giữa raw và mapping, danh mục chỉ có candidate, tổng nguồn/quán/mục, nguồn ảnh/mô tả và các trường dễ nhầm. Xuất UTF-8 BOM; đọc lại giữ chuỗi và so từng ô. Ghi đè thay vì append. Lần chạy lại với cùng snapshot/config phải cho cùng checksum CSV. Không publish seed. Nguồn mới hoặc layout đổi cần đọc report và rà tên; không để lỗi tải bị hiểu là món không tồn tại.


In [ ]:
assert set(raw_df.menu_item_id) == set(mapping_df.menu_item_id)
assert mapping_df.menu_item_id.is_unique and catalogue_df.dish_id.is_unique
accepted = mapping_df[mapping_df.mapping_status == 'candidate']
assert set(accepted.dish_id) == set(catalogue_df.dish_id)
assert catalogue_df.menu_item_count.sum() == len(accepted)
occurrence_ids = {o['occurrence_id'] for occurrences in raw_df.source_occurrences_json
                  for o in json.loads(occurrences) if o['origin'] == 'seo_menu'}
assert len(occurrence_ids) == source_df.seo_occurrences.fillna(0).sum()
assert norm('phở') != norm('phô') and norm('cốm') != norm('cơm') and norm('chạo') != norm('cháo')
base = dict(menu_groups_json='[]',venue_name='Quán thử',scope_status='selected_address_matches')
assert classify({**base,'menu_name_raw':'Cơm thêm'})['mapping_status'] == 'excluded'
assert classify({**base,'menu_name_raw':'Mắm tôm'})['mapping_status'] == 'excluded'
assert classify({**base,'menu_name_raw':'Combo 2 người'})['mapping_status'] == 'needs_review'
assert classify({**base,'menu_name_raw':'Bún chả cá'})['normalized_name'] != classify({**base,'menu_name_raw':'Bún chả'})['normalized_name']
assert classify({**base,'venue_name':'Quán Chay','menu_name_raw':'Bún bò'})['mapping_status'] == 'needs_review'
assert classify({**base,'menu_name_raw':'Hủ tiếu khô'})['dish_id'] != classify({**base,'menu_name_raw':'Hủ tiếu nước'})['dish_id']
for r in raw_df.itertuples():
    assert r.source_url in {s['url'] for s in sources}
    assert json.loads(r.source_occurrences_json)
frames = {'menu_items_raw':raw_df, 'dish_catalogue':catalogue_df, 'menu_dish_mapping':mapping_df}
checksums={}
for name, frame in frames.items():
    path=OUTPUT_PATHS[name]
    frame.to_csv(path,index=False,encoding='utf-8-sig')
    reread=pd.read_csv(path,dtype=str,keep_default_na=False,encoding='utf-8-sig')
    expected=frame.fillna('').astype(str)
    pd.testing.assert_frame_equal(reread,expected,check_dtype=False)
    assert path.read_bytes().startswith(b'\xef\xbb\xbf')
    checksums[name]=digest(path.read_bytes())
    print(path, len(reread), 'rows ×',len(reread.columns),'columns', checksums[name])
(REPORTS_DIR / 'output_checksums.json').write_text(json.dumps(checksums,indent=2)+'\n',encoding='utf-8')
display(catalogue_df.head(20))
print('Đã đối chiếu CSV sau xuất. Catalogue là draft; không chứng minh giờ trưa, tồn món hay quyền ảnh.')


## 8. Một CSV cuối chứa danh mục món và metadata nguồn

processed/thu_duc_dishes_final.csv là file để xem và dùng tiếp cho khảo sát. Mỗi tên chuẩn một dòng, bao gồm aliases, nhóm, gợi ý tìm kiếm, số quán, mô tả, URL ảnh và giá. Các cột example_* là một ví dụ từ cùng mục menu. Toàn bộ quán–món tương ứng nằm trong menu_offerings_json, giữ cả địa chỉ, nhóm menu, giá/đơn vị, ảnh/srcset, lịch/tọa độ nếu có, checksum, thời gian và lý do mapping. Trường JSON giữ metadata dạng chuỗi như raw; HTML chi tiết được giữ riêng trong raw.

Chỉ xuất candidate; excluded/needs_review vẫn nằm trong mapping và hiển thị ở cell 5. Không tạo tên mới từ nguyên liệu hay đoán các tên “Tái”, “Đặc biệt”. Thông tin chưa có giữ unknown/rỗng. Kiểm từng thuộc tính metadata về đúng nguồn và đọc lại CSV UTF-8 BOM. Run All tự ghi đè cả file cuối, không cần sửa dữ liệu giữa các cell. “Cuối” là bản xuất của lần khảo sát này, chưa là seed được reviewer duyệt hay danh sách đầy đủ toàn khu Thủ Đức.


In [ ]:
FINAL_PATH = PROCESSED_DIR / 'thu_duc_dishes_final.csv'

def build_final_catalogue(raw_frame, mapping_frame, catalogue_frame):
    accepted_mapping = mapping_frame.loc[mapping_frame.mapping_status.eq('candidate'),
        ['menu_item_id','dish_id','mapping_reason','alias_changes_json']]
    evidence = raw_frame.merge(accepted_mapping, on='menu_item_id', validate='one_to_one')
    assert len(evidence) == len(accepted_mapping)
    rows = []
    # HTML chi tiết nằm trong raw có checksum; file cuối giữ metadata và provenance.
    evidence_columns = [c for c in raw_frame.columns if c not in {'card_html','source_occurrences_json'}]
    evidence_columns += ['mapping_reason','alias_changes_json']
    for dish in catalogue_frame.to_dict('records'):
        matches = evidence[evidence.dish_id.eq(dish['dish_id'])].sort_values('menu_item_id')
        # Các cột example cùng một mục nguồn, tránh trộn giá/ảnh/mô tả giữa quán.
        choices = matches.assign(_image=matches.image_url.ne(''),
                                 _description=matches.description_raw.ne(''),
                                 _price=matches.price_value.ne('')).sort_values(
            ['_image','_description','_price','menu_item_id'], ascending=[False,False,False,True])
        example = choices.iloc[0]
        rows.append(dict(
            dish_id=dish['dish_id'], dish_name=dish['canonical_name'],
            root_dish=dish['root_dish'], category=dish['category'],
            aliases_json=dish['aliases_json'], size_variants_json=dish['size_variants_json'],
            search_queries_json=as_json([dish['canonical_name']+' Thủ Đức']),
            venue_count=dish['venue_count'], menu_item_count=dish['menu_item_count'],
            survey_clusters_json=as_json(sorted(set(matches.survey_cluster))),
            example_menu_item_id=example.menu_item_id, example_menu_name=example.menu_name_raw,
            example_venue_name=example.venue_name, example_address=example.address,
            example_description=example.description_raw, example_price_value=example.price_value,
            example_currency=example.currency, example_price_unit=example.price_unit,
            example_image_url=example.image_url, example_source_url=example.source_url,
            source_urls_json=dish['source_urls_json'],
            menu_offerings_json=as_json(matches[evidence_columns].to_dict('records')),
            images_json=dish['images_json'], descriptions_json=dish['descriptions_json'],
            first_checked_at=matches.checked_at.min(), last_checked_at=matches.checked_at.max(),
            cuisine=dish['cuisine'], origin=dish['origin'], temperature=dish['temperature'],
            flavor=dish['flavor'], meal_slots=dish['meal_slots'], availability='unknown',
            image_usage_rights='unknown', review_status='draft',
            dataset_version='thu-duc-menu-survey-final-v1'))
    if not rows:
        raise ValueError('Không có món đủ điều kiện xuất; xem mapping và nguồn.')
    return pd.DataFrame(rows).sort_values(['venue_count','dish_name'],
                                         ascending=[False,True]).reset_index(drop=True)

final_df = build_final_catalogue(raw_df, mapping_df, catalogue_df)
assert final_df.dish_id.is_unique and final_df.dish_name.is_unique
assert set(final_df.dish_id) == set(catalogue_df.dish_id)
assert int(final_df.menu_item_count.sum()) == int(mapping_df.mapping_status.eq('candidate').sum())
raw_by_id = raw_df.set_index('menu_item_id').to_dict('index')
seen_menu_ids = set()
for dish in final_df.to_dict('records'):
    offerings = json.loads(dish['menu_offerings_json'])
    assert len(offerings) == int(dish['menu_item_count'])
    assert len({o['venue_id'] for o in offerings}) == int(dish['venue_count'])
    assert sorted({o['source_url'] for o in offerings}) == json.loads(dish['source_urls_json'])
    for offering in offerings:
        mid = offering['menu_item_id']
        assert mid not in seen_menu_ids
        seen_menu_ids.add(mid)
        original = raw_by_id[mid]
        for key, value in original.items():
            if key not in {'card_html','source_occurrences_json'}:
                assert offering[key] == value, (mid, key)
        assert offering['review_status'] == 'draft'
        assert offering['scope_status'] == 'selected_address_matches'
    example = next(o for o in offerings if o['menu_item_id'] == dish['example_menu_item_id'])
    for target, origin in {'example_menu_name':'menu_name_raw', 'example_venue_name':'venue_name',
                           'example_address':'address','example_description':'description_raw',
                           'example_price_value':'price_value','example_currency':'currency',
                           'example_price_unit':'price_unit','example_image_url':'image_url',
                           'example_source_url':'source_url'}.items():
        assert dish[target] == example[origin], target
assert seen_menu_ids == set(mapping_df.loc[mapping_df.mapping_status.eq('candidate'),'menu_item_id'])

final_df.to_csv(FINAL_PATH, index=False, encoding='utf-8-sig')
final_reread = pd.read_csv(FINAL_PATH, dtype=str, keep_default_na=False, encoding='utf-8-sig')
pd.testing.assert_frame_equal(final_reread, final_df.fillna('').astype(str), check_dtype=False)
assert FINAL_PATH.read_bytes().startswith(b'\xef\xbb\xbf')
checksums['thu_duc_dishes_final'] = digest(FINAL_PATH.read_bytes())
(REPORTS_DIR / 'output_checksums.json').write_text(json.dumps(checksums, indent=2)+'\n', encoding='utf-8')
stats.update(final_dishes=len(final_df), final_columns=len(final_df.columns),
             final_menu_items=int(final_df.menu_item_count.sum()),
             final_venues=int(len({o['venue_id'] for s in final_df.menu_offerings_json for o in json.loads(s)})),
             final_dishes_with_images=int(final_df.images_json.ne('[]').sum()),
             final_dishes_with_descriptions=int(final_df.descriptions_json.ne('[]').sum()),
             source_fetch_counts=source_df.fetch_state.value_counts().to_dict(),
             final_csv=str(FINAL_PATH), final_sha256=checksums['thu_duc_dishes_final'])
(REPORTS_DIR / 'quality_report.json').write_text(json.dumps(stats, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
display(final_df[['dish_name','category','venue_count','menu_item_count','example_price_value',
                  'example_venue_name','example_description','example_image_url']].head(30))
print(FINAL_PATH)
print(len(final_df), 'món ×', len(final_df.columns), 'cột;', stats['final_menu_items'], 'mục menu có nguồn.')
print('SHA-256:', stats['final_sha256'])
print('Chưa đọc được menu:', stats['selected_sources_without_menu'], 'nguồn; tên còn cần rà:',
      stats['mapping_counts'].get('needs_review', 0))
